# Stage 9 — interactive visualization

Reusable preparation is in `src/`; Napari layers and manual diagnostic controls remain notebook concerns.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks" or PROJECT_ROOT.parent.name == "notebooks":
    while PROJECT_ROOT.name != "notebooks":
        PROJECT_ROOT = PROJECT_ROOT.parent
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
from src.io import PipelinePaths

SAMPLE_ID = "44b6_0113de3b"
FRAME = 0
paths = PipelinePaths.discover(PROJECT_ROOT)
sample_path = paths.sample_zarr(SAMPLE_ID)


In [ ]:
import pandas as pd
from src.api import prepare_visualization_data
from src.io import load_npy_time_series, load_processed_dataset_inputs, load_stage8_outputs, open_sample

inputs = load_processed_dataset_inputs(SAMPLE_ID, paths=paths)
raw = open_sample(sample_path)
outputs = load_stage8_outputs(paths=paths)
cells = pd.concat(
    [frame.assign(frame=index) for index, frame in enumerate(inputs.time_frames)],
    ignore_index=True,
)
visualization = prepare_visualization_data(outputs.tracks, cells)
preprocessed, _ = load_npy_time_series(inputs.root / "preprocessing")
mask, _ = load_npy_time_series(inputs.root / "masking")
labels, _ = load_npy_time_series(inputs.root / "segmentation")


In [ ]:
import napari
from diagnostics.cell_volume_extraction import add_cell_volume_extractor
from diagnostics.tracking_scene_extraction import add_tracking_scene_extractor

viewer = napari.Viewer(ndisplay=3)
viewer.add_image(preprocessed, name="preprocessed", colormap="gray")
viewer.add_labels(mask, name="binary mask", visible=False)
viewer.add_labels(labels, name="instances", visible=False)
viewer.add_tracks(visualization.tracks_array, name="tracks", tail_length=20)
viewer.add_points(
    visualization.points_array,
    name="track points",
    properties={"track_id": visualization.track_ids},
    size=4,
    visible=False,
)


In [ ]:
scene_extractor = add_tracking_scene_extractor(
    viewer=viewer,
    cells=cells,
    instance_labels_volume=labels,
    binary_mask_volume=mask,
    image_volumes={"raw": raw, "preprocessed": preprocessed},
    sample_id=SAMPLE_ID,
    save_root=paths.tracking_scenes,
    voxel_size_zyx=(1.625, 0.40625, 0.40625),
)
cell_extractor = add_cell_volume_extractor(
    viewer=viewer,
    cells=cells,
    image_volume=raw,
    sample_id=SAMPLE_ID,
    preprocessed_volume=preprocessed,
    binary_mask_volume=mask,
    instance_labels_volume=labels,
)
viewer
